In [1]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

2024-10-04 15:15:31.961116: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-04 15:15:32.285377: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2024-10-04 15:15:32.622994: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-10-04 15:15:32.640693: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-10-04 15:15:32.642744: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from keras.callbacks import ModelCheckpoint, EarlyStopping
from keras.src.legacy.preprocessing.image import ImageDataGenerator
import os
from sklearn.metrics import classification_report, accuracy_score

In [3]:
# Mapear arquivos ocultos (.ipny)
def filter_hidden_folders(folder_list):
    return [folder for folder in folder_list if not folder.startswith('.')]

In [4]:
# Caminho para os dados do dataset e o reescalador de imagens

#xray_directory = 'font_results/normal'
xray_directory = 'font_results/cropped'

xray_classes = filter_hidden_folders(os.listdir(xray_directory))  # Filtra pastas ocultas
os.listdir(xray_directory)

['g',
 'V',
 'W',
 'r',
 'i',
 'a',
 'G',
 'I',
 's',
 'J',
 'y',
 'z',
 't',
 'c',
 'Q',
 'u',
 'E',
 'o',
 'H',
 'B',
 'U',
 'N',
 'F',
 'm',
 'n',
 'v',
 'T',
 'D',
 'A',
 'e',
 'w',
 'O',
 '.ipynb_checkpoints',
 'S',
 'K',
 'l',
 'C',
 'q',
 'p',
 'Z',
 'j',
 'h',
 'f',
 'L',
 'M',
 'b',
 'd',
 'P',
 'x',
 'X',
 'Y',
 'R',
 'k']

In [5]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models

# Definir parâmetros
input_shape = (120, 100, 3)  # Imagem 120x100, 1 canal (grayscale)
num_classes = 53  # 26 letras maiúsculas, 26 minúsculas, + 1 "null"

# Carregar o modelo VGG16 pré-treinado com pesos do ImageNet
# Incluir a camada de pooling global (retirar as camadas de classificação final)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(120, 100, 3))

# Adaptar a camada de entrada para grayscale (1 canal)
# Expanda o canal para 3 (de 1 para 3) para o modelo pré-treinado que usa RGB
inputs = layers.Input(shape=input_shape)
x = layers.Conv2D(3, (3, 3), padding='same')(inputs)  # Converter 1 canal para 3

# Conectar o modelo pré-treinado
x = base_model(x)
x = layers.GlobalAveragePooling2D()(x)

# Adicionar as camadas de classificação
x = layers.Dense(512, activation='relu')(x)
x = layers.Dense(num_classes, activation='softmax')(x)

# Criar o modelo final
model = models.Model(inputs, x)

# Congelar as camadas do modelo pré-treinado para não treiná-las no início
for layer in base_model.layers:
    layer.trainable = False

# Compilar o modelo
model.compile(optimizer='adam', 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])

# Resumo do modelo
model.summary()



2024-10-04 15:15:35.808250: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-10-04 15:15:35.809466: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-10-04 15:15:35.810476: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 120, 100, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 120, 100, 3)    │            30 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 3, 3, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 53)             │        27,189 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,004,563 (57.24 MB)

 Trainable params: 289,875 (1.11 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [6]:
'''# Definindo o modelo
model = Sequential()

# Primeira camada de convolução
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(120, 100, 1)))  # Grayscale images, so 1 channel
model.add(MaxPooling2D(pool_size=(2, 2)))

# Segunda camada de convolução
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Terceira camada de convolução
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# Quarta camada de convolução
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

# GlobalAveragePooling2D para reduzir a dimensionalidade
model.add(GlobalAveragePooling2D())

# Camada Flatten
model.add(Flatten())

# Camadas Densas
model.add(Dense(256, activation='relu'))

model.add(Dropout(0.35))  # Dropout para evitar overfitting (desativar neuronios)

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.35))

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.35))

model.add(Dense(32, activation='relu'))
model.add(Dropout(0.35))

# Camada de Saída
model.add(Dense(53, activation='softmax'))  # 26 letras maisculas + 26 letras minusculas + 1 para "nada"
'''

'# Definindo o modelo\nmodel = Sequential()\n\n# Primeira camada de convolução\nmodel.add(Conv2D(32, (3, 3), activation=\'relu\', input_shape=(120, 100, 1)))  # Grayscale images, so 1 channel\nmodel.add(MaxPooling2D(pool_size=(2, 2)))\n\n# Segunda camada de convolução\nmodel.add(Conv2D(64, (3, 3), activation=\'relu\'))\nmodel.add(MaxPooling2D(pool_size=(2, 2)))\n\n# Terceira camada de convolução\nmodel.add(Conv2D(128, (3, 3), activation=\'relu\'))\nmodel.add(MaxPooling2D(pool_size=(2, 2)))\n\n# Quarta camada de convolução\nmodel.add(Conv2D(128, (3, 3), activation=\'relu\'))\nmodel.add(MaxPooling2D(pool_size=(2, 2)))\n\n# GlobalAveragePooling2D para reduzir a dimensionalidade\nmodel.add(GlobalAveragePooling2D())\n\n# Camada Flatten\nmodel.add(Flatten())\n\n# Camadas Densas\nmodel.add(Dense(256, activation=\'relu\'))\n\nmodel.add(Dropout(0.35))  # Dropout para evitar overfitting (desativar neuronios)\n\nmodel.add(Dense(128, activation=\'relu\'))\nmodel.add(Dropout(0.35))\n\nmodel.add(Den

In [7]:
# Compilando o modelo
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Resumo do modelo
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 120, 100, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 120, 100, 3)    │            30 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 3, 3, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 53)             │        27,189 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,004,563 (57.24 MB)

 Trainable params: 289,875 (1.11 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [8]:
# Salvar o arquivo de treinamento em diferentes epocas

# Nome/caminho do arquivo que sera salvo
filepath = "weights.keras" 
#filepath = "/keras/normal/weights.keras" 

# Define os parametros para atualizar o arquivo final
# ModelCheckpoint(nome_do_arquivo, monitor = 'parametro a ser alalisado', verbose = 1 (imprimir no console quando arquivo for salvo), save_best_only = True, mode = 'min' (objetivo minimizar perda))
checkpoint = ModelCheckpoint(filepath, monitor = 'loss', verbose = 1, save_best_only = True, mode = 'min')

# Cria uma lista que contém o ModelCheckpoint configurado usado durante o treinamento
callbacks_list = [checkpoint]

In [23]:
# Gerenciador de imagem para pre processamento
# Aumentar a base de dados 

train_datagen = ImageDataGenerator(
# Transforma os valores de pixel de (0-255 ->0.0 - 1.0 )  
    rescale=1. / 255,
# Aplica zoom aleatório às imagens dentro do intervalo especificado (0.2 -> 20%)
    zoom_range=0.2,
# Realiza uma inversão horizontal aleatória das imagens
    horizontal_flip=True,
#Este parâmetro é usado para dividir automaticamente os dados em conjuntos de treinamento e validação 
    validation_split=0.9
    )


In [10]:
# Cria critério de parada precoce
early_stopping = EarlyStopping(monitor='loss', patience=3, verbose=1)

In [20]:

BATCH_SIZE = 13
TARGET_SIZE=(120, 100)
CMODE='grayscale'
CLASS_MODE='categorical'
# Imagens para treinamento
train_generator = train_datagen.flow_from_directory(
    xray_directory,
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=CMODE, 
    class_mode=CLASS_MODE,
    shuffle=True,
    subset='training'
)

# Imagens para validação
validation_generator = train_datagen.flow_from_directory(
        xray_directory,
        target_size=TARGET_SIZE,
        batch_size=40,
        color_mode=CMODE,
        class_mode=CLASS_MODE,
        shuffle=True,
        subset = 'validation'
        )
print(train_generator.class_indices)



Found 38012 images belonging to 53 classes.
Found 3759760 images belonging to 53 classes.
{'.ipynb_checkpoints': 0, 'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7, 'H': 8, 'I': 9, 'J': 10, 'K': 11, 'L': 12, 'M': 13, 'N': 14, 'O': 15, 'P': 16, 'Q': 17, 'R': 18, 'S': 19, 'T': 20, 'U': 21, 'V': 22, 'W': 23, 'X': 24, 'Y': 25, 'Z': 26, 'a': 27, 'b': 28, 'c': 29, 'd': 30, 'e': 31, 'f': 32, 'g': 33, 'h': 34, 'i': 35, 'j': 36, 'k': 37, 'l': 38, 'm': 39, 'n': 40, 'o': 41, 'p': 42, 'q': 43, 'r': 44, 's': 45, 't': 46, 'u': 47, 'v': 48, 'w': 49, 'x': 50, 'y': 51, 'z': 52}


In [16]:
'''
checkpoint_loss = ModelCheckpoint(
    filepath='best_model_loss.keras',  # Arquivo onde o modelo será salvo
    monitor='val_loss',  # Monitora a perda de validação
    verbose=1,
    save_best_only=True,  # Salva apenas o melhor modelo
    mode='min'  # Queremos a menor perda
)
'''

"\ncheckpoint_loss = ModelCheckpoint(\n    filepath='best_model_loss.keras',  # Arquivo onde o modelo será salvo\n    monitor='val_loss',  # Monitora a perda de validação\n    verbose=1,\n    save_best_only=True,  # Salva apenas o melhor modelo\n    mode='min'  # Queremos a menor perda\n)\n"

In [21]:
# Testar o gerador de dados
for data_batch, labels_batch in train_generator:
    print("Imagens: ", data_batch.shape)
    print("Labels: ", labels_batch.shape)
    break  # Apenas testar um batch


Imagens:  (13, 120, 100, 1)
Labels:  (13, 53)


steps_per_epoch = len(train_generator) - 1  # Total de batches por época

# treinamento do modelo 


In [22]:
model = model.fit(train_generator, epochs = 3, validation_data=validation_generator, callbacks=callbacks_list) 

history = model.history.keys() #paramestros de avaliação accuracy, loss

Epoch 1/3
2920/2924 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5187 - loss: 1.6455

I0000 00:00:1728066396.625175  283930 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_222', 8 bytes spill stores, 8 bytes spill loads




Epoch 1: loss improved from inf to 1.53733, saving model to weights.keras
2924/2924 ━━━━━━━━━━━━━━━━━━━━ 1384s 473ms/step - accuracy: 0.5188 - loss: 1.6453 - val_accuracy: 0.2457 - val_loss: 4.1093
Epoch 2/3
2920/2924 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6244 - loss: 1.2660
Epoch 2: loss improved from 1.53733 to 1.22284, saving model to weights.keras
2924/2924 ━━━━━━━━━━━━━━━━━━━━ 1450s 496ms/step - accuracy: 0.6244 - loss: 1.2660 - val_accuracy: 0.2769 - val_loss: 4.0314
Epoch 3/3
2921/2924 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6785 - loss: 1.0638
Epoch 3: loss improved from 1.22284 to 1.03755, saving model to weights.keras
2924/2924 ━━━━━━━━━━━━━━━━━━━━ 1395s 477ms/step - accuracy: 0.6785 - loss: 1.0638 - val_accuracy: 0.2856 - val_loss: 4.2613


In [ ]:
# Exemplo de grafico para ver taxa de acerto por taxa de erro 
plt.plot(history.history['accuracy'])
plt.plot(history.history['loss'])
plt.title('Erro e taxa de acerto durante o treinamento')
plt.xlabel('Época')
plt.ylabel('Taxa de acerto e erro')
plt.legend(['Taxa de acerto', 'Erro']);

In [ ]:
# defini caminho para pasta de teste
test_directory = 'dataset/test'
# lista contendo os nomes dos arquivos e diretórios presentes no caminho fornecido
os.listdir(test_directory)

In [ ]:
# Mudar a escala das imagens
test_gen = ImageDataGenerator(rescale=1./255)

test_generator = test_gen.flow_from_directory(batch_size = 13, directory = test_directory,
                                              shuffle = True, target_size = (256, 256),
                                              class_mode = 'categorical')

evaluate = model.evaluate(test_generator)

len(os.listdir(test_directory))

In [ ]:
# aplica a rede treinada em um conjunto de teste de imagens e armazena os resultados
prediction = []
original = []
image = []

# percorre as subpastas no diretório test_directory, assumindo que cada subpasta contém imagens de uma determinada classe
for i in range(len(os.listdir(test_directory))):
  # percorre os arquivos dentro de cada subpasta e processa cada imagem
  for item in os.listdir(os.path.join(test_directory, str(i))):
    
    #Etapas de processamento de img

    img = cv2.imread(os.path.join(test_directory, str(i), item))
    img = cv2.resize(img, (256, 256))
    image.append(img)
    img = img / 255
    img = img.reshape(-1, 256, 256, 3)

    #rede neural aplicada
    predict = model.predict(img)
    predict = np.argmax(predict)
    prediction.append(predict)
    original.append(i)

In [ ]:
accuracy_score(original, prediction)

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(12,12))
axes = axes.ravel()
for i in np.arange(0, 25):
  axes[i].imshow(image[i])
  axes[i].set_title('Previsão={}\nTrue={}'.format(str(labels_names[prediction[i]]), str(labels_names[original[i]])))
  axes[i].axis('off')
plt.subplots_adjust(wspace = 1.2)

In [ ]:
labels_names = {0: 'SoldasCorretas', 1: 'SoldasErradas'}

In [ ]:
from keras.models import load_model

# Carregar o modelo previamente salvo
model_load = load_model('result_test/weights.keras')

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Carregar a imagem e convertê-la para um array numpy
image = np.array(Image.open("/home/picg/TCC/GeradorBase/font_results/cropped/a/a-mas-GrandHotel-Regular.png"))
#image1 = np.array(Image.open("a-23x12.png"))


# Redimensionar a imagem para o tamanho esperado pela CNN (supondo 256x256)
temp_resized = cv2.resize(image, (120, 100))

print(temp_resized.shape)

temp_resized = cv2.cvtColor(temp_resized, cv2.COLOR_BGR2RGB)


# Normalizar os valores dos pixels para [0, 1]
normalized = temp_resized / 255.0

# Adicionar uma dimensão para corresponder ao formato de entrada esperado pelo modelo
final = normalized.reshape(1, 120, 100, 3)  # '1' aqui é o batch size

# Fazer a predição com o modelo carregado
prediction = model_load.predict(final)

# Exibir o resultado da predição
print("Predição:", prediction)


In [ ]:
plt.imshow(temp_resized)

In [ ]:
#image = np.array(Image.open("/home/picg/TCC/GeradorBase/font_results/normal/a/a-mam-GrandHotel-Regular.png"))
plt.imshow(image, cmap="gray")

In [ ]:
import numpy as np

# Achar a classe com maior probabilidade
predicted_class = np.argmax(prediction)
'''
{'.ipynb_checkpoints': 0, 'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7, 'H': 8, 'I': 9, 'J': 10, 'K': 11, 'L': 12, 'M': 13, 'N': 14, 'O': 15, 'P': 16, 'Q': 17, 'R': 18, 'S': 19, 'T': 20, 'U': 21, 'V': 22, 'W': 23, 'X': 24, 'Y': 25, 'Z': 26, 'a': 27, 'b': 28, 'c': 29, 'd': 30, 'e': 31, 'f': 32, 'g': 33, 'h': 34, 'i': 35, 'j': 36, 'k': 37, 'l': 38, 'm': 39, 'n': 40, 'o': 41, 'p': 42, 'q': 43, 'r': 44, 's': 45, 't': 46, 'u': 47, 'v': 48, 'w': 49, 'x': 50, 'y': 51, 'z': 52}
'''

# Definir labels (saidas) 
labels_names = {0: '.ipynb_checkpoints', 
                1: 'a',
                2: 'A',
                3: 'b',
                4: 'B',
                5: 'c',
                6: 'C',
                7: 'd',
                8: 'D',
                9: 'e',
                10: 'E', 
                11: 'f',
                12: 'F',
                13: 'g',
                14: 'G',
                15: 'h',
                16: 'H',
                17: 'i',
                18: 'I',
                19: 'j',
                20: 'J', 
                21: 'k',
                22: 'K',
                23: 'l',
                24: 'L',
                25: 'm',
                26: 'M',
                27: 'n',
                28: 'N',
                29: 'o',
                30: 'O', 
                31: 'p',
                32: 'P',
                33: 'q',
                34: 'Q',
                35: 'r',
                36: 'R',
                37: 's',
                38: 'S',
                39: 't',
                40: 'T', 
                41: 'u',
                42: 'U',
                43: 'v',
                44: 'V',
                45: 'w',
                46: 'W',
                47: 'x',
                48: 'X',
                49: 'y',
                50: 'Y', 
                51: 'z',
                52: 'Z',
                }

print(f"A classe prevista é - {predicted_class}: {labels_names[predicted_class]}")


In [ ]:
model.summary()